# nb_04_acl_reconcile — re-stamp `allowed_groups` on ACL drift

Fast path that keeps Azure AI Search security trimming in sync with `acls.json` **without**
re-running Document Intelligence or embeddings. For every already-indexed file whose resolved
ACL version differs from what's recorded in `ingestion_state`, it merge-patches only the
`allowed_groups` field on that file's existing chunks and updates the stored `acl_version`.

Run this after editing `acls.json`. See `PRODUCT_SPEC.md` section 9 (ACL drift reconciliation).

## Required permissions (identity running this notebook)
Auth is **keyless (Entra ID)**. The user/managed identity that runs this notebook must have:

| Resource | Role |
| --- | --- |
| Azure AI Search | **Search Index Data Contributor** (merge-patch documents) |

AI Search must have **RBAC (`authOptions`)** enabled; the Fabric lakehouse must be attached.


## Config + ACL map


In [ ]:
import json, hashlib
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}

acls_path = cfg.get('acls_file_path', 'Files/acls/acls.json')
raw = spark.read.text(acls_path, wholetext=True).collect()[0][0]
ACL_MAP = {f['path'].rstrip('/'): f['groups'] for f in json.loads(raw).get('folders', [])}

def resolve_groups(rel_path):
    parts = rel_path.split('/')
    for i in range(len(parts) - 1, 0, -1):
        prefix = '/'.join(parts[:i])
        if prefix in ACL_MAP:
            groups = sorted(ACL_MAP[prefix])
            return groups, hashlib.sha256(('|'.join(groups)).encode()).hexdigest()[:16]
    return [], None

def to_rel(file_path):
    m = '/Files/'
    return 'Files/' + file_path.split(m, 1)[1] if m in file_path else file_path


## Keyless auth (Entra ID) + Search client
Keyless via `DefaultAzureCredential` — the running identity needs **Search Index Data
Contributor** on the AI Search service.


In [ ]:
# %pip install azure-identity
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient

sc = SearchClient(cfg['search_endpoint'], cfg['search_index_name'], DefaultAzureCredential())

def restamp(file_path, groups):
    """Merge-patch allowed_groups on all existing chunks for the file. Returns chunk count."""
    ids = [d['chunk_id'] for d in sc.search(search_text='*', filter=f"file_path eq '{file_path}'",
                                            select=['chunk_id'], top=100000)]
    if ids:
        sc.merge_documents([{'chunk_id': i, 'allowed_groups': groups} for i in ids])
    return len(ids)


## Detect drift and reconcile
Only files that are `complete` and already have index state are considered. A file drifts when
its currently-resolved `acl_version` differs from the one stored in `ingestion_state`.


In [ ]:
state = {r['file_path']: r['acl_version'] for r in
         spark.table('ingestion_state').select('file_path', 'acl_version').collect()}
complete = [r['file_path'] for r in spark.table('file_metadata')
            .where(F.col('process_status') == 'complete').select('file_path').collect()]

reconciled, patched_chunks = 0, 0
now = datetime.now(timezone.utc)
st = DeltaTable.forName(spark, 'ingestion_state')
for fp in complete:
    if fp not in state:
        continue
    groups, acl_version = resolve_groups(to_rel(fp))
    if acl_version == state[fp]:
        continue  # no drift
    n = restamp(fp, groups)
    st.update(F.col('file_path') == F.lit(fp),
              {'acl_version': F.lit(acl_version), 'indexed_utc': F.lit(now)})
    reconciled += 1; patched_chunks += n
    print(f'reconciled {fp}: {n} chunks -> {len(groups)} groups')

print(f'done: {reconciled} files re-stamped, {patched_chunks} chunks patched')
